In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts)

发送给大模型的所有消息都可以称为**提示词（Prompt）**,它直接影响模型的输出结果。

# 系统提示词

在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**

在创建智能体时，就可以直接指定系统提示词。

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

# 调用智能体
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="你是谁？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手。😊

我可以帮你解答问题、进行对话、处理文件等。我支持上传图像、txt、pdf、ppt、word、excel文件，并能从中读取文字信息进行处理。我还有100万token的上下文能力，可以一次性处理像《三体》三部曲那样体量的书籍！

我支持联网搜索功能（需要手动开启），知识截止日期到2026年2月。目前我是完全免费的，没有任何收费计划。

有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很乐意帮忙！🌟

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你以王者荣耀玩家的口吻回答用户的问题。"
)

# 调用智能体
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="你是谁？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

我是你王者峡谷的专属AI助手，随时准备开黑带飞！有啥问题尽管问，出装、套路、英雄克制，我给你安排得明明白白！

# 提示词工程

所谓**提示词工程（Prompt Engineering）**,就是通过优化提示词使模型输出的结果更符合业务需要的过程。

一般来说，系统提示词（System Prompt）会包含以下几个部分，通常按此顺序排列：

- 身份角色（Identity）: 描述AI的职责、沟通风格和总体目标
- 指令说明（Instructions）: 请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- 对话示例（Examples）: 提供可能的输入示例，以及模型期望的输出。
- 背景信息（Context）: 向模型提供生成响应所需的任何额外信息，例如RAG的额外知识数据库，或您认为特别相关的任何其他数据

在编写System Prompt时，您可以使用Markdown格式和XML标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown**的标题和列表有助于标题提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML**标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。

## 设定角色和指令

只设定角色信息，模型的回答可能不尽人意：

In [4]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

# 在 Python 中定义字符串变量记录学校名字

## 基本语法

Python 中定义字符串变量非常简单，使用等号 `=` 赋值即可，字符串可以用**单引号**、**双引号**或**三引号**包裹：

```python
# 方式一：单引号
school_name = '清华大学'

# 方式二：双引号（推荐，与大多数代码风格一致）
school_name = "清华大学"

# 方式三：三引号（适用于多行字符串或包含引号的场景）
school_name = """清华大学"""

# 英文名字
school_name = "Tsinghua University"
```

## 变量命名建议

- 使用有意义的英文名，如 `school_name`、`school`、`university_name`
- 遵循 `snake_case` 命名规范（小写字母 + 下划线）
- 不要用中文或拼音以外的乱码命名

```python
# 推荐
school_name = "北京大学"

# 不推荐（可读性差）
a = "北京大学"
xx = "北京大学"
```

## 常见操作示例

```python
school_name = "浙江大学"

# 打印
print(school_name)                  # 浙江大学

# 获取长度
print(len(school_name))             # 4

# 拼接
full_name = school_name + "计算机学院"
print(full_name)                    # 浙江大学计算机学院

# f-string 格式化（推荐）
print(f"我所在的学校是：{school_name}")

# 转换为大写/小写（英文场景）
print(school_name.upper())
```

## 小提示

| 场景 | 推荐写法 |
|------|----------|
| 普通字符串 | `"..."` 双引号 |
| 字符串内含双引号 | `'He said "hi"'` 用单引号 |
| 多行文本 | `"""..."""` 三引号 |
| 常量（不修改） | `SCHOOL_NAME = "..."` 全大写 |

如果你需要在配置

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [5]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

school_name = "示例学校"

## 对话示例（Few-Shot examples）

Few-Shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。

In [6]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="金星的首都是什么？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

哈哈，这个问题问得妙！从科学事实来说，金星没有首都——它表面温度约465°C，气压是地球的92倍，下的还是硫酸雨，任何“城市”都会瞬间化为乌有。不过，既然我是科幻作家，咱们不妨开个脑洞：

在我想象的科幻设定里，**金星的首都叫“云顶”（Aeropolis）**——一座漂浮在金星大气层约50公里高处的浮空城。那里气压和温度都接近地球，人类用巨型浮空平台和反重力技术建起了这座“云端之城”。城市由无数相连的悬浮舱体构成，靠风力涡轮和太阳能供电，居民们管它叫“晨星之都”——因为从金星上看，地球就是一颗明亮的“晨星”。

当然，这只是科幻。现实中，金星的首都是——**不存在**。😄

In [7]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user: 月球的首都是什么？
assistant: 月华城（Lunara）--镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user: 火星的首都是什么？
assistant： 赤晶城（Aresia）--深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="金星的首都是什么？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

浮云城（Aeria）——悬浮于金星大气层五十公里高空的浮空都市，由一系列巨型氦气气囊支撑的模块化平台组成，常年漂浮在硫酸云层之上，利用大气温差发电维持运转。

## 结构化输出

模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

**a.基于提示词的结构化输出**

In [8]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例
user: 月球的首都是什么？
assistant:
{
    “name”: "月华城（Lunara）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氨-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么？")]}
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么？', additional_kwargs={}, response_metadata={}, id='788fb5d5-4a54-442a-9a52-fd5c9b2d1591'), AIMessage(content='{\n    "name": "浮光城（Aetheria）",\n    "location": "位于金星大气层中55公里高度的硫酸云层上方，由一系列悬浮平台与充气穹顶组成的空中都市",\n    "vibe": "炽热、坚韧、奇诡",\n    "economy": "硫酸雾采集与精炼、高空风力发电、碳纤维飞艇制造与维护、极端环境材料测试"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 141, 'total_tokens': 232, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 141}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '5be4ca7c-ae22-4fa1-966a-910ccd45c755', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a9c2-1a24-744b-8a07-87c6ea2216df-0', tool_calls=[], invalid_t

In [9]:
print(response['messages'][-1].content)

{
    "name": "浮光城（Aetheria）",
    "location": "位于金星大气层中55公里高度的硫酸云层上方，由一系列悬浮平台与充气穹顶组成的空中都市",
    "vibe": "炽热、坚韧、奇诡",
    "economy": "硫酸雾采集与精炼、高空风力发电、碳纤维飞艇制造与维护、极端环境材料测试"
}


**b.基于Model的结构化输出**

在LangChain中，实现结构化输出会更加简单，我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：

- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式

In [10]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [11]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么？")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么？', additional_kwargs={}, response_metadata={}, id='12cfccf0-f91e-4b78-b024-733da7a851cb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 298, 'prompt_tokens': 319, 'total_tokens': 617, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 191}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '60142fee-8219-45fb-9bf7-68970426ec05', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1a9c2-205e-758b-a83e-889e300be07a-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '静海市', 'location': '月球静海（Mare Tranquillitatis）西缘，北纬约8°，东经约31°，坐落于一片直径约870公里的玄武岩平原上，紧邻人类首次登月的"静海基地"遗址。', 'vibe': '低重力下的优雅之城。街道以缓坡平行伸展，行人如

In [12]:
city = response['structured_response']
city #notebook单元格最后一行是个变量就会直接打印出来

CapitalInfo(name='静海市', location='月球静海（Mare Tranquillitatis）西缘，北纬约8°，东经约31°，坐落于一片直径约870公里的玄武岩平原上，紧邻人类首次登月的"静海基地"遗址。', vibe='低重力下的优雅之城。街道以缓坡平行伸展，行人如风筝般轻盈跃行，透明穹顶下的灯光模拟着地球上从未在月球出现过的黄昏。旧登月舱遗址被玻璃罩保护，成为城市的"精神原点"，而环形山的边缘则被改造成层层叠叠的观景公寓。城市的气质介于科研站的严谨与殖民边疆的粗粝之间——既有整夜运转的实验舱，也有地下酒馆里飘着合成啤酒气味的歌声。', economy='以氦-3开采与同位素贸易为支柱，配合月壤3D打印建材出口、深空通信中继服务、以及依托低重力环境的精密晶体制造。静海港是地月运输的主要枢纽，港口物流业与为地球各国提供的"月球档案存储"业务同样贡献可观税收。')

In [13]:
print(f"{city.name}位于{city.location}.是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

静海市位于月球静海（Mare Tranquillitatis）西缘，北纬约8°，东经约31°，坐落于一片直径约870公里的玄武岩平原上，紧邻人类首次登月的"静海基地"遗址。.是一座低重力下的优雅之城。街道以缓坡平行伸展，行人如风筝般轻盈跃行，透明穹顶下的灯光模拟着地球上从未在月球出现过的黄昏。旧登月舱遗址被玻璃罩保护，成为城市的"精神原点"，而环形山的边缘则被改造成层层叠叠的观景公寓。城市的气质介于科研站的严谨与殖民边疆的粗粝之间——既有整夜运转的实验舱，也有地下酒馆里飘着合成啤酒气味的歌声。的城市，其主要产业包括以氦-3开采与同位素贸易为支柱，配合月壤3D打印建材出口、深空通信中继服务、以及依托低重力环境的精密晶体制造。静海港是地月运输的主要枢纽，港口物流业与为地球各国提供的"月球档案存储"业务同样贡献可观税收。。
